# Notebook 30 - Remedy demonstration and trigger ablation (G8)

Answers the two experimental gaps from the deep review, with criteria fixed before any result exists (stage 2).

**Definitions used throughout.** A *normalisation collapse* is an epoch whose benign escalation exceeds 10% under stored BatchNorm statistics but not under batch statistics: the weights are fine, the stored statistics are not. *Degradation* is an epoch above 10% under both modes: the model is genuinely bad, for example undertrained. Only the first is the phenomenon under study, and the two are counted separately everywhere.

**Part A, does the remedy work?** NB28 showed collapsed checkpoints are healthy under batch statistics. That is evidence for the mechanism, not the remedy. Here the running statistics are reset and re-estimated on held-out training data disjoint from the recovery subset, with no gradient and no optimiser, and the model is then evaluated *normally*. Every weight tensor is asserted unchanged. A healthy checkpoint is recalibrated too, to show no harm.

**Part B, what triggers the collapse?** The four cells that collapsed in NB27/28 are rerun under one recipe change at a time: cosine learning-rate decay, a ten-times lower constant rate, a four-times smaller batch, and a ten-times lower BatchNorm momentum. The notebook stops if baseline fails to reproduce a normalisation collapse. A condition that removes collapse by failing to train is reported as exactly that.

**Part C, does a scheduled recipe shrink recovery-seed variance?** Five seeds of one method under baseline and cosine decay; the across-seed spread ratio is reported with a bootstrap interval. Descriptive, no gate.

Frozen structures from the 17b/20b registries. No new selection. No test access. Every training stage is resumable. GPU required.

**Stages.** 1 bootstrap, 2 pre-registration, 3 data, teachers, structures and dual-mode helpers with runtime probe proofs, 4 recovery-under-condition helper, 5 Part B baseline with per-epoch snapshots, 6 Part A remedy, 7 Part B remaining conditions, 8 Part C, 9 verdict, 10 figures.

In [ ]:
# Stage 1 - bootstrap
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, sys, json, copy, math
from pathlib import Path
import numpy as np, pandas as pd, torch, torch.nn as nn, yaml
import matplotlib.pyplot as plt

REPO = Path("/content/drive/MyDrive/IoT_Trust_Research/iot-trust-compression")
assert REPO.exists(), f"repo not found: {REPO}"
os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from src.saber.bridge_ciciot import load_bridge
from src.saber.taxonomy import ciciot2023_taxonomy, DEFAULT_COST_PROFILES
from src.saber.surgery import prune_cnn1d_channels
from src.saber.metrics import full_model_audit, action_weighted_boundary_inversion_rate

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
R = REPO / "results/saber"
OUT = R / "30_remedy_and_trigger"
OUT.mkdir(parents=True, exist_ok=True)
SNAP = OUT / "snapshots"; SNAP.mkdir(exist_ok=True)
print("repo:", REPO, "| device:", DEVICE)


In [ ]:
# Stage 2 - pre-registration
#
# Three questions, criteria fixed before any result exists.
#
# A. Does the proposed remedy work?  Reset BatchNorm running statistics with a forward pass
#    over held-out data (optimiser off), then evaluate NORMALLY. This is what a deployment
#    would do; batch-statistic evaluation (NB28) is not.
# B. What triggers the collapse?  Rerun the collapsing cells under recipe changes, one factor
#    at a time, with the dual-mode audit at every epoch.
# C. Does a scheduled recipe shrink recovery-seed variance?  The paper's seed-dominance claim
#    rests on a constant-learning-rate recipe; test whether cosine decay changes the spread.

PREREG = {
    "arm": "G8_remedy_and_trigger",
    "collapse_threshold_benign_to_attack": 0.10,
    "A_remedy": {
        "procedure": ("reset running mean/var, forward 50 batches of held-out training data "
                      "(disjoint from the recovery subset) in train mode with cumulative "
                      "averaging and no gradient, then evaluate in eval mode"),
        "A1": "post-recalibration eval-mode benign_to_attack <= 0.05 on every reproduced collapse",
        "A2": "post-recalibration eval-mode family macro-F1 >= 0.95 x the batch-statistics value",
        "A3": ("recalibrating a HEALTHY epoch changes benign_to_attack by <= 0.01 and family "
               "macro-F1 by <= 0.02, so the remedy does no harm where none is needed")},
    "definitions": {
        "normalisation_collapse": ("an epoch with eval-mode benign_to_attack > 0.10 whose "
                                   "batch-statistics benign_to_attack is <= 0.10: bad only under "
                                   "stored statistics"),
        "degradation": ("an epoch with benign_to_attack > 0.10 under BOTH evaluation modes: the "
                        "model is genuinely bad, e.g. undertrained; NOT counted as a collapse")},
    "B_trigger": {
        "target_cells": ["shallow/fisher/s307", "shallow/saber_v2/s307",
                         "deep/fisher/s401", "deep/saber_v2/s401"],
        "conditions": {
            "baseline":        "Adam 1e-3 constant, batch 1024, BN momentum 0.1 (reproduces NB27/28)",
            "cosine_decay":    "Adam 1e-3 -> 1e-5 cosine over all steps, batch 1024",
            "lr_1e-4":         "Adam 1e-4 constant, batch 1024",
            "batch_256":       "Adam 1e-3 constant, batch 256",
            "bn_momentum_0.01":"Adam 1e-3 constant, batch 1024, BN momentum 0.01"},
        "outcome": "count of NORMALISATION collapses per condition across the four cells; degradation "
                   "epochs reported separately",
        "eliminates": "zero normalisation collapses in all four cells",
        "reduces": "fewer normalisation collapses than baseline",
        "trains_adequately": "median final family macro-F1 >= 0.90 x the baseline value; a condition that "
                             "eliminates collapse by failing to train is reported as such",
        "note": "baseline must reproduce at least one collapse or the ablation is uninformative"},
    "C_seed_variance": {
        "design": "shallow, fisher, 5 seeds [101,211,307,401,503], 8 units on the 10% subset, "
                  "baseline vs cosine_decay",
        "outcome": "SD across seeds of final-unit AWBIR, and normalisation-collapse count, per condition",
        "reporting": "descriptive ratio with a bootstrap 95% interval; no pass/fail gate"},
    "no_test_access": True, "no_new_selection": True,
}
(OUT / "G8_PREREGISTRATION.json").write_text(json.dumps(PREREG, indent=2))
print(json.dumps(PREREG, indent=2))

COLLAPSE_B2A = PREREG["collapse_threshold_benign_to_attack"]
SEEDS_C = [101, 211, 307, 401, 503]
E_MAX = {"shallow": 8, "deep": 6}
SUBSET_FRACTION = 0.10
CONDITIONS = {
    "baseline":         dict(lr=1e-3, schedule=None,     batch=1024, bn_momentum=0.1),
    "cosine_decay":     dict(lr=1e-3, schedule="cosine", batch=1024, bn_momentum=0.1),
    "lr_1e-4":          dict(lr=1e-4, schedule=None,     batch=1024, bn_momentum=0.1),
    "batch_256":        dict(lr=1e-3, schedule=None,     batch=256,  bn_momentum=0.1),
    "bn_momentum_0.01": dict(lr=1e-3, schedule=None,     batch=1024, bn_momentum=0.01),
}
TARGETS_B = [("shallow", "fisher", 307), ("shallow", "saber_v2", 307),
             ("deep", "fisher", 401), ("deep", "saber_v2", 401)]


In [ ]:
# Stage 3 - data, teachers, frozen structures, dual-mode helpers
TRAIN_LOADER, VAL_LOADER, _TEST_UNUSED, SHALLOW_TEACHER, CLASS_NAMES = load_bridge()
taxonomy = ciciot2023_taxonomy(CLASS_NAMES)
robust_graph = pd.read_csv(R / "14_risk_graph/asvg_edges_robust.csv")
N_CLASSES = len(CLASS_NAMES)
SABER_CFG = yaml.safe_load(open(REPO / "config/saber.yaml"))
MIN_W = {"shallow": int(SABER_CFG["groups"]["minimum_remaining_per_layer"]), "deep": 8}


class DeepCNN1D(nn.Module):
    def __init__(self, n_classes=34):
        super().__init__()
        def blk(i, o):
            return [nn.Conv1d(i, o, 3, padding=1), nn.ReLU(), nn.BatchNorm1d(o)]
        self.conv = nn.Sequential(*blk(1, 64), *blk(64, 128), nn.MaxPool1d(2),
                                  *blk(128, 128), *blk(128, 256))
        self.pool = nn.AdaptiveAvgPool1d(1)
        self.head = nn.Linear(256, n_classes)

    def forward(self, x):
        if x.dim() == 2:
            x = x.unsqueeze(1)
        return self.head(self.pool(self.conv(x.float())).squeeze(-1))


TEACHERS = {"shallow": SHALLOW_TEACHER.to(DEVICE).eval()}
_dt = DeepCNN1D(N_CLASSES)
_dt.load_state_dict(torch.load(REPO / "models/ciciot2023/deepcnn1d_g5_seed0.pt",
                               map_location="cpu", weights_only=False)["state_dict"])
TEACHERS["deep"] = _dt.to(DEVICE).eval()

Xv, Yv = VAL_LOADER.dataset.tensors
VAL_Y_ALL = Yv.numpy()
_rng = np.random.default_rng(0)
_idx = np.concatenate([_rng.permutation(np.where(VAL_Y_ALL == c)[0])[:4000]
                       for c in range(N_CLASSES) if (VAL_Y_ALL == c).sum() > 0])
_idx = np.random.default_rng(12345).permutation(_idx)          # shuffled for batch-stat probing
EX_X = Xv[_idx].to(DEVICE); EX_Y = VAL_Y_ALL[_idx]
EXAMPLE_INPUT = Xv[:8].float().to(DEVICE)

_train_y = TRAIN_LOADER.dataset.tensors[1].numpy()
_counts = np.bincount(_train_y, minlength=N_CLASSES)
_w = np.zeros_like(_counts, dtype=np.float64)
_w[_counts > 0] = 1.0 / np.sqrt(_counts[_counts > 0]); _w[_counts > 0] /= _w[_counts > 0].mean()
CLASS_W = torch.tensor(_w, dtype=torch.float32, device=DEVICE)
N_TRAIN = len(TRAIN_LOADER.dataset)


def forward_logits(model):
    model.eval()
    with torch.no_grad():
        return torch.cat([model(EX_X[i:i + 8192]).cpu()
                          for i in range(0, len(EX_X), 8192)]).numpy()


def forward_logits_batch_stats(model):
    saved = {n: (m.running_mean.clone(), m.running_var.clone(), m.momentum,
                 m.num_batches_tracked.clone())
             for n, m in model.named_modules()
             if isinstance(m, nn.BatchNorm1d) and m.running_mean is not None}
    model.train()
    for n, m in model.named_modules():
        if n in saved:
            m.momentum = 0.0
    with torch.no_grad():
        out = torch.cat([model(EX_X[i:i + 8192]).cpu()
                         for i in range(0, len(EX_X), 8192)]).numpy()
    for n, m in model.named_modules():
        if n in saved:
            rm, rv, mom, nbt = saved[n]
            m.running_mean.copy_(rm); m.running_var.copy_(rv)
            m.momentum = mom; m.num_batches_tracked.copy_(nbt)
    model.eval()
    return out


T_LOGITS = {a: forward_logits(m) for a, m in TEACHERS.items()}


def audit(model, arch, mode="eval"):
    lg = forward_logits(model) if mode == "eval" else forward_logits_batch_stats(model)
    a = full_model_audit(lg, EX_Y, taxonomy, DEFAULT_COST_PROFILES)
    aw, _ = action_weighted_boundary_inversion_rate(T_LOGITS[arch], lg, EX_Y, robust_graph)
    return {"b2a": float(a["benign_to_attack_rate"]), "a2b": float(a["attack_to_benign_rate"]),
            "family_f1": float(a["family_macro_f1"]), "fine_f1": float(a["fine_macro_f1"]),
            "awbir": float(aw), "ece": float(a["ece15"])}


def bn_var_max(model):
    return float(max(m.running_var.max().item() for m in model.modules()
                     if isinstance(m, nn.BatchNorm1d)))


def removed_path(arch, method):
    if arch == "shallow":
        return R / f"17b_calibrated_checkpoint_freeze/{method}_r40cal_removed_groups.csv"
    return R / f"20b_depth_checkpoint_freeze/{method}_minimal_r40_removed_groups.csv"


def raw_student(arch, method):
    rm = pd.read_csv(removed_path(arch, method))
    pm = {str(l): sorted(g["channel_index"].astype(int).tolist())
          for l, g in rm.groupby("module_path")}
    st, _ = prune_cnn1d_channels(TEACHERS[arch], pm, EXAMPLE_INPUT,
                                 minimum_remaining_per_layer=MIN_W[arch])
    return st.to(DEVICE)


# runtime proof: the probe is non-destructive and discriminative
for _a, _m in TEACHERS.items():
    _b = {n: (x.running_mean.clone(), x.running_var.clone())
          for n, x in _m.named_modules() if isinstance(x, nn.BatchNorm1d)}
    _e1 = forward_logits(_m); _p = forward_logits_batch_stats(_m); _e2 = forward_logits(_m)
    for n, x in _m.named_modules():
        if isinstance(x, nn.BatchNorm1d):
            assert torch.equal(x.running_mean, _b[n][0]) and torch.equal(x.running_var, _b[n][1])
    assert np.allclose(_e1, _e2, atol=1e-5) and not _m.training
    assert not np.allclose(_e1, _p, atol=1e-6), f"{_a}: probe indistinguishable from eval mode"
print("probe verified on:", list(TEACHERS), "| evaluation rows:", len(_idx))


In [ ]:
# Stage 4 - recovery under a named condition, with snapshots and the dual-mode audit
def make_loaders(seed, batch):
    """Recovery subset (10%, seeded exactly as NB27/28) and a disjoint held-out calibration slice."""
    gen = torch.Generator().manual_seed(seed)
    perm = torch.randperm(N_TRAIN, generator=gen)
    n_sub = int(N_TRAIN * SUBSET_FRACTION)
    sub_idx = perm[:n_sub]
    cal_idx = perm[n_sub:n_sub + 50 * 1024]                     # disjoint from the recovery subset
    sub = torch.utils.data.DataLoader(
        torch.utils.data.Subset(TRAIN_LOADER.dataset, sub_idx.tolist()),
        batch_size=batch, shuffle=True, generator=torch.Generator().manual_seed(seed))
    cal = torch.utils.data.DataLoader(
        torch.utils.data.Subset(TRAIN_LOADER.dataset, cal_idx.tolist()),
        batch_size=1024, shuffle=False)
    return sub, cal


def set_bn_momentum(model, momentum):
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d):
            m.momentum = momentum


def recover(arch, method, seed, cond, tag, snapshot_epochs=()):
    """Train E_MAX[arch] units under CONDITIONS[cond]; return per-epoch dual-mode rows."""
    cfg = CONDITIONS[cond]
    torch.manual_seed(seed); np.random.seed(seed)
    student = raw_student(arch, method)
    set_bn_momentum(student, cfg["bn_momentum"])
    loader, _ = make_loaders(seed, cfg["batch"])
    opt = torch.optim.Adam(student.parameters(), lr=cfg["lr"])
    total_steps = E_MAX[arch] * len(loader)
    sched = (torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total_steps, eta_min=1e-5)
             if cfg["schedule"] == "cosine" else None)
    lossf = nn.CrossEntropyLoss(weight=CLASS_W)
    rows = []
    for unit in range(1, E_MAX[arch] + 1):
        student.train()
        for xb, yb in loader:
            opt.zero_grad()
            lossf(student(xb.float().to(DEVICE)), yb.to(DEVICE)).backward()
            opt.step()
            if sched is not None:
                sched.step()
        ev, bs = audit(student, arch, "eval"), audit(student, arch, "batch")
        rows.append({"tag": tag, "architecture": arch, "method": method, "seed": seed,
                     "condition": cond, "unit": unit, "lr_now": float(opt.param_groups[0]["lr"]),
                     "bn_var_max": bn_var_max(student),
                     **{f"eval_{k}": v for k, v in ev.items()},
                     **{f"batch_{k}": v for k, v in bs.items()}})
        if unit in snapshot_epochs:
            torch.save({"state_dict": student.state_dict(), "arch": arch, "method": method,
                        "seed": seed, "unit": unit, "condition": cond},
                       SNAP / f"{tag}_{arch}_{method}_s{seed}_u{unit}.pt")
    return rows


def rebuild_from_snapshot(path):
    p = torch.load(path, map_location="cpu", weights_only=False)
    st = raw_student(p["arch"], p["method"])
    st.load_state_dict(p["state_dict"])
    return st.to(DEVICE).eval(), p
print("recovery helpers ready")


In [ ]:
# Stage 5 - Part B baseline: reproduce the collapses and snapshot every epoch
BASE_CSV = OUT / "B_baseline_epochs.csv"
base_rows = pd.read_csv(BASE_CSV).to_dict("records") if BASE_CSV.exists() else []
done = {(r["architecture"], r["method"], r["seed"]) for r in base_rows}
for arch, method, seed in TARGETS_B:
    if (arch, method, seed) in done:
        continue
    rows = recover(arch, method, seed, "baseline", "B",
                   snapshot_epochs=tuple(range(1, E_MAX[arch] + 1)))
    base_rows.extend(rows)
    pd.DataFrame(base_rows).to_csv(BASE_CSV, index=False)
    hits = [r["unit"] for r in rows if r["eval_b2a"] > COLLAPSE_B2A and r["batch_b2a"] <= COLLAPSE_B2A]
    print(f"baseline {arch} {method} s{seed}: normalisation collapse at {hits or 'none'}")

base = pd.DataFrame(base_rows)
_norm = (base["eval_b2a"] > COLLAPSE_B2A) & (base["batch_b2a"] <= COLLAPSE_B2A)
_degr = (base["eval_b2a"] > COLLAPSE_B2A) & (base["batch_b2a"] > COLLAPSE_B2A)
print("baseline: normalisation collapses", int(_norm.sum()), "| degradation epochs", int(_degr.sum()))
assert _norm.sum() > 0, "baseline reproduced no normalisation collapse; the ablation would be uninformative"


In [ ]:
# Stage 6 - Part A: the remedy, applied to reproduced collapsed checkpoints
def recalibrate_bn(model, cal_loader, n_batches=50):
    """Reset running statistics and re-estimate them on held-out data. No gradient, no optimiser."""
    for m in model.modules():
        if isinstance(m, nn.BatchNorm1d):
            m.reset_running_stats()
            m.momentum = None                       # cumulative average over the pass
    model.train()
    with torch.no_grad():
        for i, (xb, _) in enumerate(cal_loader):
            if i >= n_batches:
                break
            model(xb.float().to(DEVICE))
    set_bn_momentum(model, 0.1)
    model.eval()
    return model


base = pd.read_csv(OUT / "B_baseline_epochs.csv")
collapsed = base[(base["eval_b2a"] > COLLAPSE_B2A) & (base["batch_b2a"] <= COLLAPSE_B2A)]
healthy = base[base["eval_b2a"] <= COLLAPSE_B2A]
print("normalisation collapses to treat:", len(collapsed), "| healthy epochs available:", len(healthy))
assert len(collapsed) > 0, "no normalisation collapse in baseline; rerun stage 5"

A_rows = []
# A1/A2: every reproduced collapse
for r in collapsed.itertuples():
    path = SNAP / f"B_{r.architecture}_{r.method}_s{r.seed}_u{r.unit}.pt"
    model, meta = rebuild_from_snapshot(path)
    before = audit(model, r.architecture, "eval")
    batchref = audit(model, r.architecture, "batch")
    _, cal = make_loaders(int(r.seed), 1024)
    # snapshot weights to prove recalibration touches only BN buffers
    w_before = {n: p.detach().clone() for n, p in model.named_parameters()}
    recalibrate_bn(model, cal)
    for n, p in model.named_parameters():
        assert torch.equal(p, w_before[n]), f"recalibration altered weight {n}"
    after = audit(model, r.architecture, "eval")
    A_rows.append({"kind": "collapsed", "architecture": r.architecture, "method": r.method,
                   "seed": int(r.seed), "unit": int(r.unit),
                   "b2a_before": before["b2a"], "b2a_after": after["b2a"],
                   "b2a_batchstats": batchref["b2a"],
                   "family_f1_before": before["family_f1"], "family_f1_after": after["family_f1"],
                   "family_f1_batchstats": batchref["family_f1"],
                   "awbir_before": before["awbir"], "awbir_after": after["awbir"]})
    print(f"REMEDY {r.architecture} {r.method} s{r.seed} u{r.unit}: b2a {before['b2a']:.4f} -> "
          f"{after['b2a']:.4f} | famF1 {before['family_f1']:.3f} -> {after['family_f1']:.3f} "
          f"(batch-stats ref {batchref['family_f1']:.3f})")

# A3: does the remedy harm a healthy checkpoint? one healthy epoch per cell
for (arch, method, seed), g in healthy.groupby(["architecture", "method", "seed"]):
    r = g.sort_values("unit").iloc[-1]
    path = SNAP / f"B_{arch}_{method}_s{seed}_u{int(r.unit)}.pt"
    model, _ = rebuild_from_snapshot(path)
    before = audit(model, arch, "eval")
    _, cal = make_loaders(int(seed), 1024)
    recalibrate_bn(model, cal)
    after = audit(model, arch, "eval")
    A_rows.append({"kind": "healthy_control", "architecture": arch, "method": method,
                   "seed": int(seed), "unit": int(r.unit),
                   "b2a_before": before["b2a"], "b2a_after": after["b2a"], "b2a_batchstats": np.nan,
                   "family_f1_before": before["family_f1"], "family_f1_after": after["family_f1"],
                   "family_f1_batchstats": np.nan,
                   "awbir_before": before["awbir"], "awbir_after": after["awbir"]})
    print(f"CONTROL {arch} {method} s{seed} u{int(r.unit)}: b2a {before['b2a']:.4f} -> "
          f"{after['b2a']:.4f} | famF1 {before['family_f1']:.3f} -> {after['family_f1']:.3f}")

A = pd.DataFrame(A_rows); A.to_csv(OUT / "A_remedy.csv", index=False)


In [ ]:
# Stage 7 - Part B: the other four conditions on the same cells
ABL_CSV = OUT / "B_ablation_epochs.csv"
abl_rows = pd.read_csv(ABL_CSV).to_dict("records") if ABL_CSV.exists() else []
done = {(r["condition"], r["architecture"], r["method"], r["seed"]) for r in abl_rows}
for cond in [c for c in CONDITIONS if c != "baseline"]:
    for arch, method, seed in TARGETS_B:
        if (cond, arch, method, seed) in done:
            continue
        rows = recover(arch, method, seed, cond, "B")
        abl_rows.extend(rows)
        pd.DataFrame(abl_rows).to_csv(ABL_CSV, index=False)
        hits = [r["unit"] for r in rows if r["eval_b2a"] > COLLAPSE_B2A and r["batch_b2a"] <= COLLAPSE_B2A]
        degr = [r["unit"] for r in rows if r["eval_b2a"] > COLLAPSE_B2A and r["batch_b2a"] > COLLAPSE_B2A]
        print(f"{cond:17s} {arch} {method} s{seed}: norm-collapse {hits or 'none'} degrade {degr or 'none'} | "
              f"max BN var {max(r['bn_var_max'] for r in rows):.2f}")
abl = pd.DataFrame(abl_rows)


In [ ]:
# Stage 8 - Part C: seed variance under baseline vs cosine decay (shallow, fisher, 5 seeds)
C_CSV = OUT / "C_seed_variance_epochs.csv"
c_rows = pd.read_csv(C_CSV).to_dict("records") if C_CSV.exists() else []
done = {(r["condition"], r["seed"]) for r in c_rows}
for cond in ["baseline", "cosine_decay"]:
    for seed in SEEDS_C:
        if (cond, seed) in done:
            continue
        rows = recover("shallow", "fisher", seed, cond, "C")
        c_rows.extend(rows)
        pd.DataFrame(c_rows).to_csv(C_CSV, index=False)
        hits = [r["unit"] for r in rows if r["eval_b2a"] > COLLAPSE_B2A and r["batch_b2a"] <= COLLAPSE_B2A]
        degr = [r["unit"] for r in rows if r["eval_b2a"] > COLLAPSE_B2A and r["batch_b2a"] > COLLAPSE_B2A]
        print(f"C {cond:12s} s{seed}: final awbir {rows[-1]['eval_awbir']:.4f} | "
              f"norm-collapse {hits or 'none'} degrade {degr or 'none'}")
C = pd.DataFrame(c_rows)


In [ ]:
# Stage 9 - verdict
for _f in ["B_baseline_epochs.csv", "A_remedy.csv", "B_ablation_epochs.csv",
           "C_seed_variance_epochs.csv"]:
    assert (OUT / _f).exists(), f"{_f} missing: run the earlier stages in order first"
base = pd.read_csv(OUT / "B_baseline_epochs.csv")
abl = pd.read_csv(OUT / "B_ablation_epochs.csv")
A = pd.read_csv(OUT / "A_remedy.csv")
C = pd.read_csv(OUT / "C_seed_variance_epochs.csv")
NORM = lambda g: (g["eval_b2a"] > COLLAPSE_B2A) & (g["batch_b2a"] <= COLLAPSE_B2A)
DEGR = lambda g: (g["eval_b2a"] > COLLAPSE_B2A) & (g["batch_b2a"] > COLLAPSE_B2A)

# ---- A ----
col = A[A["kind"] == "collapsed"]; ctl = A[A["kind"] == "healthy_control"]
A1 = bool(len(col) and (col["b2a_after"] <= 0.05).all())
A2 = bool(len(col) and (col["family_f1_after"] >= 0.95 * col["family_f1_batchstats"]).all())
A3 = bool(len(ctl) and ((ctl["b2a_after"] - ctl["b2a_before"]).abs() <= 0.01).all()
          and ((ctl["family_f1_after"] - ctl["family_f1_before"]).abs() <= 0.02).all())

# ---- B ----
allB = pd.concat([base, abl])
per_cond = {}
for cond, g in allB.groupby("condition"):
    n_norm, n_degr = int(NORM(g).sum()), int(DEGR(g).sum())
    finals = g.groupby(["architecture", "method", "seed"]).last()
    per_cond[cond] = {
        "normalisation_collapses": n_norm,
        "degradation_epochs": n_degr,
        "cells_with_normalisation_collapse": int(g[NORM(g)].groupby(
            ["architecture", "method", "seed"]).ngroups),
        "epoch_evaluations": int(len(g)),
        "max_bn_running_var": float(g["bn_var_max"].max()),
        "median_final_awbir": float(finals["eval_awbir"].median()),
        "median_final_family_f1": float(finals["eval_family_f1"].median())}
n_base = per_cond["baseline"]["normalisation_collapses"]
f1_base = per_cond["baseline"]["median_final_family_f1"]
for cond, v in per_cond.items():
    v["trains_adequately"] = bool(v["median_final_family_f1"] >= 0.90 * f1_base)
    if cond == "baseline":
        v["verdict"] = "baseline"
    elif v["normalisation_collapses"] == 0:
        v["verdict"] = "eliminates" if v["trains_adequately"] else "eliminates_but_undertrains"
    elif v["normalisation_collapses"] < n_base:
        v["verdict"] = "reduces"
    else:
        v["verdict"] = "does_not_reduce"

# ---- C ----
def final_awbir(frame, cond):
    g = frame[frame["condition"] == cond].sort_values("unit")
    return g.groupby("seed")["eval_awbir"].last().values

fb, fc = final_awbir(C, "baseline"), final_awbir(C, "cosine_decay")
rng = np.random.default_rng(0)
ratios = []
for _ in range(2000):
    b = rng.choice(fb, len(fb), replace=True); c = rng.choice(fc, len(fc), replace=True)
    if b.std(ddof=1) > 0:
        ratios.append(c.std(ddof=1) / b.std(ddof=1))
C_summary = {
    "final_awbir_sd_baseline": float(fb.std(ddof=1)), "final_awbir_sd_cosine": float(fc.std(ddof=1)),
    "sd_ratio_cosine_over_baseline": float(fc.std(ddof=1) / fb.std(ddof=1)),
    "sd_ratio_bootstrap_95ci": [float(np.percentile(ratios, 2.5)), float(np.percentile(ratios, 97.5))],
    "final_awbir_mean_baseline": float(fb.mean()), "final_awbir_mean_cosine": float(fc.mean()),
    "normalisation_collapses_baseline": int(NORM(C[C.condition == "baseline"]).sum()),
    "normalisation_collapses_cosine": int(NORM(C[C.condition == "cosine_decay"]).sum()),
    "degradation_epochs_baseline": int(DEGR(C[C.condition == "baseline"]).sum()),
    "degradation_epochs_cosine": int(DEGR(C[C.condition == "cosine_decay"]).sum()),
    "n_seeds": int(len(fb))}

verdict = {
    "arm": "G8_remedy_and_trigger",
    "A_remedy": {"A1_post_recalibration_b2a_le_0.05": A1,
                 "A2_family_f1_recovers_to_batchstats": A2,
                 "A3_no_harm_on_healthy": A3,
                 "n_collapses_treated": int(len(col)), "n_healthy_controls": int(len(ctl)),
                 "collapsed_b2a_before_median": float(col["b2a_before"].median()) if len(col) else None,
                 "collapsed_b2a_after_median": float(col["b2a_after"].median()) if len(col) else None,
                 "collapsed_family_f1_before_median": float(col["family_f1_before"].median()) if len(col) else None,
                 "collapsed_family_f1_after_median": float(col["family_f1_after"].median()) if len(col) else None},
    "B_trigger": per_cond,
    "C_seed_variance": C_summary,
    "prereg": json.load(open(OUT / "G8_PREREGISTRATION.json")),
}
(OUT / "G8_verdict.json").write_text(json.dumps(verdict, indent=2))
print(json.dumps({k: v for k, v in verdict.items() if k != "prereg"}, indent=2))


In [ ]:
# Stage 10 - figures
A = pd.read_csv(OUT / "A_remedy.csv"); col = A[A["kind"] == "collapsed"]
base = pd.read_csv(OUT / "B_baseline_epochs.csv"); abl = pd.read_csv(OUT / "B_ablation_epochs.csv")
C = pd.read_csv(OUT / "C_seed_variance_epochs.csv")

fig, ax = plt.subplots(figsize=(6.4, 3.4))
x = np.arange(len(col))
ax.bar(x - 0.2, col["b2a_before"], 0.4, label="collapsed checkpoint, eval mode", color="#c44e52")
ax.bar(x + 0.2, col["b2a_after"], 0.4, label="after BatchNorm recalibration", color="#55a868")
ax.set_xticks(x); ax.set_xticklabels([f"{r.architecture[0]}/{r.method}/s{r.seed}/u{r.unit}"
                                       for r in col.itertuples()], rotation=35, ha="right", fontsize=7)
ax.set_ylabel("benign to attack rate"); ax.set_yscale("symlog", linthresh=1e-3)
ax.axhline(0.05, color="0.4", ls=":", lw=0.9); ax.legend(fontsize=7)
ax.set_title("Part A: the remedy on every reproduced collapse")
fig.tight_layout(); fig.savefig(OUT / "G8_A_remedy.png", dpi=200); plt.show()

allB = pd.concat([base, abl])
conds = list(CONDITIONS)
fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.4))
def _norm(g): return int(((g["eval_b2a"] > COLLAPSE_B2A) & (g["batch_b2a"] <= COLLAPSE_B2A)).sum())
def _degr(g): return int(((g["eval_b2a"] > COLLAPSE_B2A) & (g["batch_b2a"] > COLLAPSE_B2A)).sum())
xs = np.arange(len(conds))
axes[0].bar(xs - 0.2, [_norm(allB[allB.condition == c]) for c in conds], 0.4,
            label="normalisation collapses", color="#c44e52")
axes[0].bar(xs + 0.2, [_degr(allB[allB.condition == c]) for c in conds], 0.4,
            label="degradation (bad under both modes)", color="#8c8c8c")
axes[0].set_xticks(xs); axes[0].set_xticklabels(conds, rotation=25, ha="right", fontsize=8)
axes[0].set_ylabel("epochs (4 cells)"); axes[0].legend(fontsize=7)
axes[0].set_title("Part B: collapses vs degradation by condition")
for c in conds:
    g = allB[allB.condition == c].groupby("unit")["bn_var_max"].max()
    axes[1].plot(g.index, g.values, marker="o", lw=1.2, label=c)
axes[1].set_yscale("log"); axes[1].set_xlabel("recovery unit"); axes[1].set_ylabel("max BN running var")
axes[1].set_title("Part B: worst running variance per epoch"); axes[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "G8_B_trigger.png", dpi=200); plt.show()

fig, axes = plt.subplots(1, 2, figsize=(10.4, 3.4))
for ax, cond in zip(axes, ["baseline", "cosine_decay"]):
    g = C[C.condition == cond]
    for seed in SEEDS_C:
        s = g[g.seed == seed].sort_values("unit")
        ax.plot(s["unit"], s["eval_awbir"], marker="o", lw=1.1, label=f"seed {seed}")
    ax.set_title(f"Part C: shallow / fisher / {cond}"); ax.set_xlabel("recovery unit")
    ax.set_ylabel("AWBIR (eval mode)"); ax.legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "G8_C_seed_variance.png", dpi=200); plt.show()
print("figures written ->", OUT)
